<a href="https://colab.research.google.com/github/hjiwoong/DL/blob/main/day14_practice3_XAI_GradCAM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# XAI(Explainable AI, 설명 가능한 인공지능)는 AI가 왜 그런 판단을 내렸는지 사람이 이해할 수 있도록 설명해주는 기술
# Grad-CAM (Gradient-weighted Class Activation Mapping)은 CNN이 이미지를 보고 "어느 부분을 보고 이 클래스로 판단했는가"를 시각적으로 보여주는 방법

# CNN의 Gradient와 Feature Map을 이용하여 모델의 특정 클래스 예측에 영향을 준 이미지 영역을 Heatmap으로 시각화

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F # 신경망에서 쓰는 함수를 모아둔 모듈
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cpu


In [2]:
# 셀 1. 준비
def load_fmnist(root="./data"):
  tf = transforms.ToTensor()
  tr = datasets.FashionMNIST(root=root, train=True, download=True, transform=tf)
  te = datasets.FashionMNIST(root=root, train=False, download=True, transform=tf)
  return tr, te

train_data, test_data = load_fmnist()
classes = train_data.classes

class CNN(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Sequential(nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2))
    self.conv2 = nn.Sequential(nn.Conv2d(16, 32, 3, padding=1), nn.ReLU()) # ←CAM 대상
    self.pool2 = nn.MaxPool2d(2)
    self.head = nn.Sequential(nn.Flatten(), nn.Linear(32*7*7, 128), nn.ReLU(), nn.Linear(128, 10))

  def forward(self, x):
    self.fmap = self.conv2(self.conv1(x)) # 특징맵 저장
    return self.head(self.pool2(self.fmap))

model = CNN().to(device)
loader = DataLoader(train_data, batch_size=256, shuffle=True)
loss_fn, opt = nn.CrossEntropyLoss(), torch.optim.Adam(model.parameters(), lr=0.001)
for epoch in range(2):
  for x, y in loader:
    x, y = x.to(device), y.to(device)
    loss = loss_fn(model(x), y)
    opt.zero_grad(); loss.backward(); opt.step()
print(f"준비 완료 (2 epoch, loss {loss.item():.3f})")

100%|██████████| 26.4M/26.4M [00:02<00:00, 12.8MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 201kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.77MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 7.71MB/s]


준비 완료 (2 epoch, loss 0.358)
